# analysis.ipynb — Deep Message Analysis

In [ ]:
import sqlite3, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter
from pathlib import Path
from scipy.stats import gaussian_kde

warnings.filterwarnings("ignore", message="Glyph.*missing from font")

import matplotlib.font_manager as fm
cjk_candidates = ["Microsoft JhengHei", "Microsoft YaHei", "Noto Sans CJK TC", "Arial Unicode MS"]
cjk_fonts = [f for f in cjk_candidates if any(f.lower() in x.name.lower() for x in fm.fontManager.ttflist)]
if not cjk_fonts: cjk_fonts = ["DejaVu Sans"]
plt.rcParams["font.sans-serif"] = [cjk_fonts[0], "DejaVu Sans", "Segoe UI Emoji"]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 96
plt.rcParams["savefig.dpi"] = 96

DB_PATH = str(Path(".") / "messages.db")
conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row

try:
    from contacts_data import YOUR_ALIASES
    YOUR_NAMES = [name for name, _ in YOUR_ALIASES]
except ImportError:
    YOUR_NAMES = ["Your Name"]
YOU = ", ".join(f"'{n}'" for n in YOUR_NAMES)
print(f"YOU = {YOUR_NAMES}")

from sentence_transformers import SentenceTransformer
_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model ready.")

## 1. Wrapped Stats

In [ ]:
stats = {}
r = conn.execute("SELECT COUNT(*) n, SUM(word_count) w FROM messages WHERE content_type='text'").fetchone()
stats["Total messages"] = f"{r['n']:,}"
stats["Total words"]    = f"{r['w']:,}"
r2 = conn.execute(f"SELECT COUNT(*) n FROM messages WHERE sender IN ({YOU}) AND content_type='text'").fetchone()
stats["Your messages"]  = f"{r2['n']:,}  ({r2['n']/r['n']*100:.1f}%)"
bd = conn.execute("SELECT DATE(timestamp_ms/1000,'unixepoch') d, COUNT(*) n FROM messages GROUP BY d ORDER BY n DESC LIMIT 1").fetchone()
stats["Busiest day"]    = f"{bd['d']}  ({bd['n']:,} messages)"
tc = conn.execute(f"SELECT t.thread_name, COUNT(*) n FROM messages m JOIN threads t ON m.thread_id=t.thread_id WHERE t.is_group=0 GROUP BY t.thread_name ORDER BY n DESC LIMIT 1").fetchone()
stats["Top DM contact"] = f"{tc['thread_name']}  ({tc['n']:,} messages)"
days = pd.read_sql("SELECT DISTINCT DATE(timestamp_ms/1000,'unixepoch') d FROM messages ORDER BY d", conn)["d"].tolist()
max_streak = streak = 1
for i in range(1, len(days)):
    streak = streak+1 if (pd.Timestamp(days[i])-pd.Timestamp(days[i-1])).days==1 else 1
    max_streak = max(max_streak, streak)
stats["Longest streak"] = f"{max_streak} days"
cr = conn.execute("SELECT COUNT(*) n, SUM(call_duration) total_s, MAX(call_duration) max_s FROM messages WHERE content_type='call' AND call_duration IS NOT NULL").fetchone()
if cr["n"]:
    stats["Total calls"]     = f"{cr['n']:,}"
    stats["Total call time"] = f"{cr['total_s']/3600:.1f} hrs"
    stats["Longest call"]    = f"{cr['max_s']//3600}h {(cr['max_s']%3600)//60}m"
yr = conn.execute("SELECT MIN(CAST(STRFTIME('%Y',timestamp_ms/1000,'unixepoch') AS INT)) y0, MAX(CAST(STRFTIME('%Y',timestamp_ms/1000,'unixepoch') AS INT)) y1 FROM messages").fetchone()
stats["Years active"]   = f"{yr['y0']} - {yr['y1']}  ({yr['y1']-yr['y0']+1} yrs)"
fig, ax = plt.subplots(figsize=(10, len(stats)*0.55+0.5))
ax.axis("off")
for i,(k,v) in enumerate(stats.items()):
    y = 1-(i+0.5)/len(stats)
    ax.text(0.02, y, k, ha="left",  va="center", fontsize=12, color="#888")
    ax.text(0.98, y, v, ha="right", va="center", fontsize=13, fontweight="bold", color="#222")
ax.set_title("Your Messaging Wrapped", fontsize=16, fontweight="bold", pad=10)
plt.tight_layout(); plt.show()

## 2. Communication Style per Contact

In [ ]:
TOP_STYLE = 15
EMOJI_RE = re.compile(r"[\U0001F300-\U0001FAFF\U00002600-\U000027BF\U0001F900-\U0001F9FF]", re.UNICODE)
top_contacts = pd.read_sql(
    f"SELECT t.thread_name FROM messages m JOIN threads t ON m.thread_id=t.thread_id "
    f"WHERE t.is_group=0 AND m.sender IN ({YOU}) AND m.content_type='text' "
    f"GROUP BY t.thread_name ORDER BY COUNT(*) DESC LIMIT {TOP_STYLE}", conn
)["thread_name"].tolist()

style_rows = []
for contact in top_contacts:
    df_c = pd.read_sql(
        f"SELECT m.content, m.word_count FROM messages m JOIN threads t ON m.thread_id=t.thread_id "
        f"WHERE t.thread_name=? AND m.sender IN ({YOU}) AND m.content_type='text'",
        conn, params=[contact])
    texts = df_c["content"].dropna().tolist()
    if not texts: continue
    words_all = " ".join(texts).lower().split()
    style_rows.append({
        "contact":       contact,
        "vocab_richness": len(set(words_all))/max(len(words_all),1)*100,
        "question_rate":  sum(1 for t in texts if "?" in t)/len(texts)*100,
        "emoji_rate":     sum(len(EMOJI_RE.findall(t)) for t in texts)/len(texts)*10,
        "avg_msg_len":    df_c["word_count"].mean(),
    })
df_style = pd.DataFrame(style_rows).set_index("contact")

fig, axes = plt.subplots(1, 4, figsize=(18, max(4, len(df_style)*0.4)))
cols   = ["vocab_richness","question_rate","emoji_rate","avg_msg_len"]
titles = ["Vocab richness (%)","Question rate (%)","Emoji rate (x10/msg)","Avg msg length (words)"]
colors = ["#4a90d9","#e07b4a","#f7c948","#74c476"]
for ax,col,title,color in zip(axes,cols,titles,colors):
    d = df_style[col].sort_values(ascending=False)
    ax.barh(d.index, d.values, color=color, alpha=0.85)
    ax.set_title(title, fontsize=9); ax.invert_yaxis()
    ax.tick_params(axis='y', labelsize=8)
plt.suptitle("How you write — per contact", fontsize=13)
plt.tight_layout(); plt.show()

## 3. Social Mask Map

**Axes are pre-defined semantic dimensions (not UMAP coordinates):**
- **X — Energy**: emoji rate, exclamation rate, caps ratio, reaction rate vs. avg message length
- **Y — Intent**: personal pronouns, short questions, sentiment strength, hedging vs. long questions, function words, bare URLs

Features extracted per thread → z-score normalised → composite axis scores → KMeans(6) clusters in 2D.

In [ ]:
import re
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer as _VaderSA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, QuantileTransformer

try:
    from snownlp import SnowNLP as _SnowNLP
    _SNOW_OK = True
except ImportError:
    _SNOW_OK = False
    print("snownlp not installed — CJK sentiment will be skipped (pip install snownlp)")

try:
    from contacts_data import KNOWN_CONTACTS
    _ALIAS_MAP = {alias: canonical
                  for canonical, aliases in KNOWN_CONTACTS.items()
                  for alias, _platform in aliases}
except ImportError:
    _ALIAS_MAP = {}

_vader_md  = _VaderSA()
_EMOJI_RE  = re.compile(r"[\U0001F300-\U0001FAFF\U00002600-\U000027BF\U0001F900-\U0001F9FF]", re.UNICODE)
_URL_RE    = re.compile(r"https?://\S+|www\.\S+")
_PRON_RE   = re.compile(r"\b(i|me|my|mine|we|us|our|you|your|你|我|妳|咱|我們|你們)\b", re.IGNORECASE)
_HEDGE_RE  = re.compile(r"\b(maybe|perhaps|probably|might|i think|i guess|not sure|idk|dunno|感覺|好像|應該|大概|可能)\b", re.IGNORECASE)
_FUNC_RE   = re.compile(r"\b(the|a|an|is|are|was|were|have|has|had|will|would|could|should|do|does|did|been|being)\b", re.IGNORECASE)
_CJK_RE    = re.compile(r"[一-鿿]")
# Kaomoji: western emoticons, blushing (///), and parenthesised faces like (^_^) (>_<)
_KAOMOJI_RE = re.compile(
    r'[;:=8XxB][\-o\*\']?[\)\]\(\[dDpP\/\:\}\{@\|\\]'
    r'|[\)\]\(\[dDpP\/\:\}\{@\|\\][\-o\*\']?[;:=8XxB]'
    r'|\([/\\|*]{2,}\)'
    r'|\([^)]{0,12}[_\-\^vwωoO><≧≦ᵕ][^)]{0,12}\)',
    re.UNICODE
)

_SYS_RE = re.compile(
    r"^("
    r"you missed a call|missed a call from|started a (video|audio) call|"
    r"sent an attachment|reacted .+ to your|you are now connected|"
    r"named the (group|chat)|added .+ to the (group|chat)|"
    r"left the (group|chat)|removed .+ from|changed the (group|chat)|"
    r"turned on (ephemeral|disappearing)|pinned a message|"
    r"[‎‏‪-‮]"
    r")", re.IGNORECASE
)

def _is_system(text):
    return bool(_SYS_RE.match(text.strip()))

def _sentiment_score(text):
    cjk_chars     = len(_CJK_RE.findall(text))
    non_cjk_alpha = sum(c.isalpha() for c in text) - cjk_chars
    # Per-message language routing: majority CJK → SnowNLP, else → VADER.
    # SnowNLP recentered: its chat-neutral (~0.3) maps to 0 instead of -0.4.
    if _SNOW_OK and cjk_chars > non_cjk_alpha:
        try:
            return (_SnowNLP(text).sentiments - 0.3) / 0.7
        except Exception:
            pass
    return _vader_md.polarity_scores(text)["compound"]

def _burst_aggregate(rows, gap_s=120):
    aggregated = []
    buf_sender, buf_ts, buf_texts = None, None, []
    for ts_ms, sender, content in rows:
        if _is_system(content):
            continue
        ts_s = ts_ms / 1000
        if sender == buf_sender and buf_ts is not None and (ts_s - buf_ts) <= gap_s:
            buf_texts.append(content)
            buf_ts = ts_s
        else:
            if buf_texts:
                aggregated.append("\n".join(buf_texts))
            buf_sender, buf_ts, buf_texts = sender, ts_s, [content]
    if buf_texts:
        aggregated.append("\n".join(buf_texts))
    return aggregated

# Adaptive sample size: scales with log10(total) so high-volume contacts
# get proportionally more samples without dominating at 50x the size.
# k=150 gives: 200→~150, 1k→~300, 5k→~408, 50k→~500 (capped at MAX_SAMPLE).
MIN_SAMPLE = 80
MAX_SAMPLE = 600

def _adaptive_limit(total, k=150):
    return min(MAX_SAMPLE, max(MIN_SAMPLE, int(k * np.log10(total + 1))))

N_MASKS  = 6
MIN_MSGS = 5

ENERGY_W = {"emoji_rate":+1, "kaomoji_rate":+1, "excl_rate":+1, "caps_ratio":+1, "reaction_rate":+1, "avg_len":-1}
INTENT_W = {"short_q_rate":+1, "pronoun_rate":+1, "sent_strength":+1,
            "hedge_rate":+1,  "url_emoji_rate":+1,
            "long_q_rate":-1, "func_rate":-1,     "bare_url_rate":-1}
_ALL_FEAT_COLS = list(dict.fromkeys(list(ENERGY_W) + list(INTENT_W)))

def _extract_features(texts):
    n  = len(texts)
    # avg_len: per individual message (pre-burst), not per merged burst unit.
    # Bursts are "\n"-joined so split("\n") recovers original messages.
    msgs = [seg for t in texts for seg in t.split("\n") if seg.strip()]
    wc   = [len(m.split()) for m in msgs] if msgs else [0]
    q    = [t for t in texts if "?" in t or "？" in t]
    ql   = [len(t.split()) for t in q]
    u    = [t for t in texts if _URL_RE.search(t)]
    vs   = [_sentiment_score(t) for t in texts[:150]]
    return {
        "emoji_rate":     sum(len(_EMOJI_RE.findall(t)) for t in texts) / n,
        "kaomoji_rate":   sum(len(_KAOMOJI_RE.findall(t)) for t in texts) / n,
        "excl_rate":      sum(t.count("!") for t in texts) / n,
        "caps_ratio":     sum(sum(c.isupper() for c in t) / max(len(t), 1) for t in texts) / n,
        "avg_len":        float(np.mean(wc)),
        "reaction_rate":  sum(1 for w in wc if w <= 3) / n,
        "short_q_rate":   sum(1 for l in ql if l <= 5) / n,
        "long_q_rate":    sum(1 for l in ql if l >  5) / n,
        "pronoun_rate":   sum(len(_PRON_RE.findall(t)) for t in texts) / max(sum(wc), 1),
        "hedge_rate":     sum(1 for t in texts if _HEDGE_RE.search(t)) / n,
        "func_rate":      sum(len(_FUNC_RE.findall(t)) for t in texts) / max(sum(wc), 1),
        "url_emoji_rate": sum(1 for t in u if _EMOJI_RE.search(t)) / n,
        "bare_url_rate":  sum(1 for t in u if not _EMOJI_RE.search(t)) / n,
        "sent_strength":  float(np.mean(np.abs(vs))),
    }

def _fetch_and_burst(thread_ids, sender="you", limit=None):
    """
    Fetches ALL matching messages, burst-aggregates, then stratified-samples
    evenly across the full time range.
    limit=None → adaptive log-scaled size based on total bursted count.
    limit=N    → fixed cap (use for per-year drift or explicit overrides).
    Returns (texts, total_bursted_count).
    """
    ph = ",".join("?" * len(thread_ids))
    if sender == "you":
        sender_clause = f"AND sender IN ({YOU})"
    elif sender == "them":
        sender_clause = f"AND sender NOT IN ({YOU})"
    else:
        sender_clause = ""
    rows = conn.execute(
        f"SELECT timestamp_ms, sender, content FROM messages "
        f"WHERE thread_id IN ({ph}) {sender_clause} "
        f"AND content_type='text' AND LENGTH(content)>5 "
        f"ORDER BY timestamp_ms",
        thread_ids
    ).fetchall()
    bursted = _burst_aggregate(rows)
    total   = len(bursted)
    n       = limit if limit is not None else _adaptive_limit(total)
    if total <= n:
        return bursted, total
    rng    = np.random.default_rng(42)
    bucket = total / n
    sampled = [bursted[int(rng.integers(int(i * bucket), max(int((i+1) * bucket), int(i*bucket)+1)))]
               for i in range(n)]
    return sampled, total

def _raw_composite(df_f, scaler):
    Z = pd.DataFrame(scaler.transform(df_f[_ALL_FEAT_COLS]), columns=_ALL_FEAT_COLS)
    e = sum(Z[f] * s for f, s in ENERGY_W.items())
    i = sum(Z[f] * s for f, s in INTENT_W.items())
    return e.values, i.values

def _axis_scores(df_f, scaler, qt_e, qt_i):
    e_raw, i_raw = _raw_composite(df_f, scaler)
    e_q = qt_e.transform(e_raw.reshape(-1, 1)).ravel() * 2 - 1
    i_q = qt_i.transform(i_raw.reshape(-1, 1)).ravel() * 2 - 1
    return e_q, i_q

# ── Per-identity feature extraction ───────────────────────────────────────────
threads = pd.read_sql("SELECT thread_id, thread_name, is_group FROM threads", conn)

identity_groups = {}
for _, row in threads.iterrows():
    label = row["thread_name"] if row["is_group"] else _ALIAS_MAP.get(row["thread_name"], row["thread_name"])
    if label not in identity_groups:
        identity_groups[label] = {"is_group": int(row["is_group"]), "thread_ids": []}
    identity_groups[label]["thread_ids"].append(row["thread_id"])

feat_rows = []
for label, meta in identity_groups.items():
    texts, total = _fetch_and_burst(meta["thread_ids"], sender="you")  # adaptive limit
    if len(texts) < MIN_MSGS:
        continue
    f = _extract_features(texts)
    f.update({"thread_name": label, "is_group": meta["is_group"],
               "n_msgs": len(texts), "n_msgs_total": total})
    feat_rows.append(f)

df_feats = pd.DataFrame(feat_rows).reset_index(drop=True)
print(f"Identities: {len(df_feats)}  ({(df_feats.is_group==0).sum()} DMs, {(df_feats.is_group==1).sum()} GCs)")
print("\nTop 10 DMs by real message count (with adaptive sample size):")
top10 = df_feats[df_feats.is_group==0].nlargest(10, "n_msgs_total")[["thread_name","n_msgs_total","n_msgs","kaomoji_rate","emoji_rate","avg_len"]]
print(top10.to_string(index=False))

# ── Fit scalers ────────────────────────────────────────────────────────────────
_scaler = StandardScaler().fit(df_feats[_ALL_FEAT_COLS])
_e_raw, _i_raw = _raw_composite(df_feats, _scaler)
_qt_e = QuantileTransformer(output_distribution="uniform", random_state=42).fit(_e_raw.reshape(-1, 1))
_qt_i = QuantileTransformer(output_distribution="uniform", random_state=42).fit(_i_raw.reshape(-1, 1))

df_feats["energy"], df_feats["intent"] = _axis_scores(df_feats, _scaler, _qt_e, _qt_i)

# ── KMeans ─────────────────────────────────────────────────────────────────────
_kmeans_2d = KMeans(n_clusters=N_MASKS, random_state=42, n_init=10)
df_feats["mask"] = _kmeans_2d.fit_predict(df_feats[["energy","intent"]].values)

MASK_COLORS = plt.cm.Set2(np.linspace(0, 1, N_MASKS))
cluster_ids = list(range(N_MASKS))

print("\n── Mask Clusters ─────────────────────────────────────────────────────────")
for ci in sorted(cluster_ids, key=lambda c: _kmeans_2d.cluster_centers_[c, 1], reverse=True):
    cx, cy = _kmeans_2d.cluster_centers_[ci]
    sub = df_feats[df_feats["mask"] == ci].sort_values("n_msgs_total", ascending=False)
    print(f"\n[Mask {ci}]  {'high-energy' if cx>0 else 'restrained'} × {'connect' if cy>0 else 'content/task'}   n={len(sub)}")
    for _, r in sub.head(5).iterrows():
        tag = " [GC]" if r["is_group"] else ""
        print(f"    {r['thread_name']}{tag}   (e={r['energy']:.2f}, i={r['intent']:.2f}, msgs={int(r['n_msgs_total'])}, kaomoji={r['kaomoji_rate']:.3f})")

In [ ]:
# ── EDIT THIS after reading the representatives above ─────────────────────────
# Map cluster index → your human-readable mask label
MASK_NAMES = {
    # 0: "Leisure / BS",
    # 1: "Emotional / Support",
    # 2: "Planning / Logistics",
    # 3: "Hype / Reactions",
    # ...
}
# ──────────────────────────────────────────────────────────────────────────────

mask_labels = [MASK_NAMES.get(i, f"Mask {i}") for i in cluster_ids]
n_masks = len(mask_labels)
MASK_COLORS = plt.cm.Set2(np.linspace(0, 1, n_masks))

print("Current mask labels:")
for i, lbl in enumerate(mask_labels):
    print(f"  {i}: {lbl}")

## 3a. Style Profile per Social Mask

Each mask cluster implicitly encodes a social context (work GCs cluster together, friend GCs cluster together, etc.). Use mask assignment as context label and compare how your writing style shifts across them.

In [ ]:
_AFFIRM_RE  = re.compile(r"\b(ok|okay|yeah|yep|yes|sure|alright|gotcha|got it|noted|對|好|嗯|好的|好喔|知道了|了解)\b", re.IGNORECASE)
_COMPLEX_RE = re.compile(r"[,;:，；：]")
_FILLER_RE  = re.compile(r"\b(like|basically|literally|actually|honestly|tbh|lol|haha|哈|欸|啊|喔|哦|嗎|吧)\b", re.IGNORECASE)

def _rich_style(texts):
    if not texts:
        return None
    n   = len(texts)
    # Per-message word count (pre-burst), consistent with _extract_features avg_len
    msgs = [seg for t in texts for seg in t.split("\n") if seg.strip()]
    wc   = [len(m.split()) for m in msgs] if msgs else [0]
    all_words   = " ".join(msgs).lower().split()
    total_chars = sum(len(t) for t in texts)
    return {
        "vocab_richness":      len(set(all_words)) / max(len(all_words), 1),
        "avg_len":             float(np.mean(wc)),
        "cjk_ratio":           sum(len(_CJK_RE.findall(t)) for t in texts) / max(total_chars, 1),
        "emoji_rate":          sum(len(_EMOJI_RE.findall(t)) for t in texts) / n,
        "kaomoji_rate":        sum(len(_KAOMOJI_RE.findall(t)) for t in texts) / n,
        "excl_rate":           sum(t.count("!") for t in texts) / n,
        "question_rate":       sum(1 for t in texts if "?" in t or "？" in t) / n,
        "func_rate":           sum(len(_FUNC_RE.findall(t)) for t in texts) / max(sum(wc), 1),
        "affirmation_rate":    sum(1 for t in texts if _AFFIRM_RE.search(t)) / n,
        "punctuation_density": sum(len(_COMPLEX_RE.findall(t)) for t in texts) / max(sum(wc), 1),
        "filler_rate":         sum(1 for t in texts if _FILLER_RE.search(t)) / n,
        "reaction_rate":       sum(1 for w in wc if w <= 3) / n,
    }

_STYLE_FEAT_LABELS = {
    "vocab_richness":      "Vocab richness",
    "avg_len":             "Msg length",
    "cjk_ratio":           "Chinese ratio",
    "emoji_rate":          "Emoji rate",
    "kaomoji_rate":        "Kaomoji rate",
    "excl_rate":           "Exclamation rate",
    "question_rate":       "Question rate",
    "func_rate":           "Formality",
    "affirmation_rate":    "Affirmation rate",
    "punctuation_density": "Sentence complexity",
    "filler_rate":         "Filler / casual words",
    "reaction_rate":       "Short reactions",
}
_SFCOLS = list(_STYLE_FEAT_LABELS.keys())

# ── Pool messages per mask — adaptive sample across all threads in that mask ──
mask_style_rows = []
for ci in cluster_ids:
    thread_names = df_feats[df_feats["mask"] == ci]["thread_name"].tolist()
    tids = [tid for tn in thread_names for tid in identity_groups.get(tn, {}).get("thread_ids", [])]
    if not tids:
        continue
    texts, _ = _fetch_and_burst(tids, sender="you")  # adaptive
    sv = _rich_style(texts)
    if sv is None:
        continue
    sv["mask"]          = ci
    sv["label"]         = mask_labels[ci]
    sv["n_threads"]     = len(thread_names)
    sv["n_msgs_sample"] = len(texts)
    mask_style_rows.append(sv)

df_mstyle = pd.DataFrame(mask_style_rows)

# ── Radar chart ───────────────────────────────────────────────────────────────
df_norm = df_mstyle[_SFCOLS].copy()
for col in _SFCOLS:
    lo, hi = df_norm[col].min(), df_norm[col].max()
    df_norm[col] = (df_norm[col] - lo) / max(hi - lo, 1e-9)

N_feat = len(_SFCOLS)
angles = np.linspace(0, 2 * np.pi, N_feat, endpoint=False).tolist() + [0]

fig, ax_r = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
ax_r.set_theta_offset(np.pi / 2)
ax_r.set_theta_direction(-1)

for idx, row in df_mstyle.iterrows():
    vals  = df_norm.loc[idx, _SFCOLS].tolist() + [df_norm.loc[idx, _SFCOLS[0]]]
    color = MASK_COLORS[int(row["mask"])]
    ax_r.plot(angles, vals, color=color, lw=2, label=f"{row['label']}  (n={row['n_threads']})")
    ax_r.fill(angles, vals, color=color, alpha=0.07)

ax_r.set_xticks(angles[:-1])
ax_r.set_xticklabels([_STYLE_FEAT_LABELS[f] for f in _SFCOLS], fontsize=8)
ax_r.set_yticks([0.25, 0.5, 0.75, 1.0])
ax_r.set_yticklabels(["25%", "50%", "75%", "100%"], fontsize=6, color="#aaa")
ax_r.set_ylim(0, 1)
ax_r.legend(loc="upper right", bbox_to_anchor=(1.35, 1.12), fontsize=8)
ax_r.set_title("Style Profile per Social Mask\n(normalised — shows relative shift, not absolute values)", fontsize=11, pad=20)
plt.tight_layout(); plt.show()

# ── Bar breakdown per feature ─────────────────────────────────────────────────
n_cols = 4
n_rows = int(np.ceil(len(_SFCOLS) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, n_rows * 2.8))
axes = axes.ravel()

for fi, feat in enumerate(_SFCOLS):
    ax_f = axes[fi]
    vals = [df_mstyle.loc[df_mstyle["mask"]==ci, feat].values[0]
            if ci in df_mstyle["mask"].values else 0 for ci in cluster_ids]
    ax_f.bar(range(len(cluster_ids)), vals, color=[MASK_COLORS[ci] for ci in cluster_ids], alpha=0.85)
    ax_f.set_xticks(range(len(cluster_ids)))
    ax_f.set_xticklabels([mask_labels[ci] for ci in cluster_ids], fontsize=6.5, rotation=20, ha="right")
    ax_f.set_title(_STYLE_FEAT_LABELS[feat], fontsize=8)
    ax_f.tick_params(axis="y", labelsize=7)

for fi in range(len(_SFCOLS), len(axes)):
    axes[fi].set_visible(False)

plt.suptitle("Style breakdown per mask — raw values", fontsize=11)
plt.tight_layout(); plt.show()

### 3a. Reference Mask Matching

Match each discovered cluster to a curated library of social masks drawn from psychology and communication literature (Goffman's dramaturgy, Jung's persona theory, interpersonal circumplex, register theory). Matching is done by dot product between each cluster's z-scored feature vector and each reference mask's prototype weights.

In [ ]:
# ── Reference mask library ────────────────────────────────────────────────────
# Sources: Goffman (1959) dramaturgy; Jung persona theory; Leary interpersonal
#   circumplex; CAT (Giles 1973); register theory (Halliday); relational work
#   (Locher & Watts 2005); digital communication styles (Herring 2007)

REF_MASKS = {
    # ── High-energy × connect ────────────────────────────────────────────────
    "Hype Man": {
        "emoji_rate": +2, "excl_rate": +2, "reaction_rate": +2,
        "filler_rate": +1, "avg_len": -1, "func_rate": -1,
        "energy": +2, "intent": +1,
        "_note": "Goffman front-stage performer; high expressive display, brief reactive turns"
    },
    "The Bestie": {
        "emoji_rate": +1, "filler_rate": +2, "affirmation_rate": +2,
        "cjk_ratio": +1, "avg_len": -0.5, "question_rate": +1,
        "energy": +1, "intent": +2,
        "_note": "Jung intimate alter ego; code-switching, high validation loops"
    },
    # ── Restrained × connect ─────────────────────────────────────────────────
    "The Confidant": {
        "question_rate": +1, "affirmation_rate": +1, "avg_len": +1,
        "filler_rate": +1, "func_rate": -1, "reaction_rate": -0.5,
        "energy": -0.5, "intent": +2,
        "_note": "Locher & Watts relational work; emotionally receptive, low-energy high-warmth"
    },
    "The Listener": {
        "affirmation_rate": +2, "question_rate": +2, "reaction_rate": +1,
        "excl_rate": -1, "avg_len": -0.5, "emoji_rate": -0.5,
        "energy": -1, "intent": +1,
        "_note": "Interpersonal circumplex — high agreeableness, low dominance; draws out others"
    },
    # ── High-energy × task ───────────────────────────────────────────────────
    "The Coordinator": {
        "question_rate": +2, "affirmation_rate": +1, "excl_rate": +1,
        "avg_len": +0.5, "func_rate": +0.5, "emoji_rate": -0.5,
        "energy": +1, "intent": -1,
        "_note": "CAT task register; directive questions, confirmation-seeking turns"
    },
    "The Teammate": {
        "emoji_rate": +1, "affirmation_rate": +2, "question_rate": +1,
        "filler_rate": +1, "func_rate": -0.5, "avg_len": -0.5,
        "energy": +1, "intent": -0.5,
        "_note": "Warm-task blend; social glue while executing logistics"
    },
    # ── Restrained × task ────────────────────────────────────────────────────
    "The Advisor": {
        "avg_len": +2, "vocab_richness": +2, "punctuation_density": +2,
        "func_rate": +1, "emoji_rate": -2, "filler_rate": -2,
        "energy": -1, "intent": -2,
        "_note": "Halliday formal tenor; high lexical density, elaborated code"
    },
    "The Collaborator": {
        "question_rate": +1, "affirmation_rate": +2, "avg_len": +1,
        "func_rate": +0.5, "filler_rate": -1, "reaction_rate": -0.5,
        "energy": -0.5, "intent": -0.5,
        "_note": "Structured reciprocity; task-oriented with warm acknowledgement loops"
    },
    # ── Edge personas ─────────────────────────────────────────────────────────
    "The Intellectual": {
        "vocab_richness": +3, "punctuation_density": +2, "avg_len": +2,
        "func_rate": +1, "filler_rate": -2, "emoji_rate": -1,
        "energy": -1, "intent": -1,
        "_note": "Bernstein elaborated code; high lexical diversity, complex sentences"
    },
    "The Lurker": {
        # Distinguishing signal: very short turns, low energy, near-zero initiation
        # Must NOT fire for high-energy clusters — add explicit energy penalty
        "reaction_rate": +3, "avg_len": -3, "filler_rate": -2,
        "emoji_rate": -1, "excl_rate": -2, "affirmation_rate": +1,
        "energy": -3, "intent": 0,
        "_note": "Minimal footprint; mostly acks and short reactions, never initiates"
    },
    "The Performer": {
        "emoji_rate": +2, "excl_rate": +2, "vocab_richness": +1,
        "filler_rate": +1, "avg_len": +0.5, "affirmation_rate": -0.5,
        "energy": +2, "intent": 0,
        "_note": "Goffman impression management; expressive, self-presenting, broadcast-mode"
    },
    "The Caretaker": {
        "question_rate": +2, "affirmation_rate": +2, "avg_len": +1,
        "emoji_rate": +1, "filler_rate": +0.5, "func_rate": -0.5,
        "energy": 0, "intent": +2,
        "_note": "High communal orientation; checks in, validates, initiates social bonding"
    },
}

# ── Build cluster feature matrix ───────────────────────────────────────────────
_centers = pd.DataFrame(_kmeans_2d.cluster_centers_, columns=["energy", "intent"])
_centers["mask"] = cluster_ids

df_match = df_mstyle.merge(_centers, on="mask")
_MATCH_COLS = _SFCOLS + ["energy", "intent"]

_match_z = df_match[_MATCH_COLS].copy()
for col in _MATCH_COLS:
    mu, sd = _match_z[col].mean(), _match_z[col].std()
    _match_z[col] = (_match_z[col] - mu) / max(sd, 1e-9)

# ── Score each cluster against each reference mask ────────────────────────────
_ref_names = [k for k in REF_MASKS if not k.startswith("_")]
score_mat = np.zeros((len(df_match), len(_ref_names)))

for ri, ref_name in enumerate(_ref_names):
    proto = REF_MASKS[ref_name]
    for ci_idx in range(len(df_match)):
        score = sum(
            float(_match_z.iloc[ci_idx][feat]) * w
            for feat, w in proto.items()
            if feat != "_note" and feat in _match_z.columns
        )
        score_mat[ci_idx, ri] = score

# ── Report ─────────────────────────────────────────────────────────────────────
print("── Reference Mask Matching ────────────────────────────────────────────────")
_SUGGESTED = {}
for ci_idx, row in df_match.iterrows():
    ci = int(row["mask"])
    cx, cy = _kmeans_2d.cluster_centers_[ci]
    ranked = sorted(zip(_ref_names, score_mat[ci_idx]), key=lambda x: -x[1])
    _SUGGESTED[ci] = ranked[0][0]
    print(f"\n[Mask {ci}]  {'high-energy' if cx>0 else 'restrained'} × {'connect' if cy>0 else 'content/task'}  n={int(row['n_threads'])}")
    for ref, sc in ranked[:3]:
        print(f"  {sc:+.2f}  {ref:25s}  ↳ {REF_MASKS[ref].get('_note','')[:70]}")

# ── Heatmap ───────────────────────────────────────────────────────────────────
fig, ax_m = plt.subplots(figsize=(len(_ref_names) * 0.9 + 1, len(df_match) * 0.7 + 1))
im = ax_m.imshow(score_mat, aspect="auto", cmap="RdYlGn",
                 vmin=-score_mat.std()*2, vmax=score_mat.std()*2)
ax_m.set_xticks(range(len(_ref_names)))
ax_m.set_xticklabels(_ref_names, rotation=35, ha="right", fontsize=8)
ax_m.set_yticks(range(len(df_match)))
ax_m.set_yticklabels([f"Mask {int(r['mask'])}  ({mask_labels[int(r['mask'])]})" for _, r in df_match.iterrows()], fontsize=8)
for ci_idx in range(len(df_match)):
    best_ri = int(score_mat[ci_idx].argmax())
    ax_m.add_patch(plt.Rectangle((best_ri-0.5, ci_idx-0.5), 1, 1, fill=False, edgecolor="#333", lw=2))
plt.colorbar(im, ax=ax_m, label="Alignment score  (green = strong match)")
ax_m.set_title("Cluster × Reference Mask Alignment\n□ = best match per cluster", fontsize=10)
plt.tight_layout(); plt.show()

print("\n── Suggested MASK_NAMES ──────────────────────────────────────────────────")
print("MASK_NAMES = {")
for ci, suggestion in sorted(_SUGGESTED.items()):
    print(f"    {ci}: \"{suggestion}\",")
print("}")

In [ ]:
mask_labels = [MASK_NAMES.get(i, f"Mask {i}") for i in cluster_ids]

fig, ax = plt.subplots(figsize=(13, 10))
ax.axhline(0, color="#ddd", lw=0.8, zorder=0)
ax.axvline(0, color="#ddd", lw=0.8, zorder=0)

ax_lim = float(df_feats[["energy","intent"]].abs().values.max()) * 1.1

for (sx, sy, lbl) in [(1,1,"high-energy + connect"), (-1,1,"restrained + connect"),
                       (1,-1,"high-energy + task"),  (-1,-1,"restrained + task")]:
    ax.text(ax_lim*sx*0.55, ax_lim*sy*0.95, lbl,
            ha="center", fontsize=7.5, color="#bbb", style="italic")

for ci, mcolor in enumerate(MASK_COLORS):
    pts = df_feats[df_feats["mask"]==ci][["energy","intent"]].values
    if len(pts) < 4:
        continue
    try:
        kde = gaussian_kde(pts.T, bw_method=0.6)
        margin = ax_lim * 0.12
        xi = np.linspace(-ax_lim-margin, ax_lim+margin, 160)
        yi = np.linspace(-ax_lim-margin, ax_lim+margin, 160)
        Xi, Yi = np.meshgrid(xi, yi)
        Zi = kde(np.vstack([Xi.ravel(), Yi.ravel()])).reshape(Xi.shape)
        ax.contourf(Xi, Yi, Zi, levels=[Zi.max()*0.10, Zi.max()], colors=[mcolor], alpha=0.15)
        ax.contour( Xi, Yi, Zi, levels=[Zi.max()*0.20], colors=[mcolor], linewidths=0.9, alpha=0.55)
        pk = np.unravel_index(Zi.argmax(), Zi.shape)
        ax.text(Xi[pk], Yi[pk], mask_labels[ci],
                ha="center", va="center", fontsize=9, color=mcolor, fontweight="bold", alpha=0.8)
    except Exception:
        pass

for ci, (mlabel, mcolor) in enumerate(zip(mask_labels, MASK_COLORS)):
    for is_grp, mkr in [(0, "o"), (1, "s")]:
        sub = df_feats[(df_feats["mask"]==ci) & (df_feats["is_group"]==is_grp)]
        if sub.empty:
            continue
        sz = np.clip(np.sqrt(sub["n_msgs_total"].values) * 1.5, 30, 400)
        ax.scatter(sub["energy"], sub["intent"], c=[mcolor], s=sz, marker=mkr,
                   alpha=0.9, edgecolors="white", linewidths=0.8, zorder=5)

# Labels: top-10 DMs by real count + all GCs
for _, r in df_feats[df_feats["is_group"]==0].nlargest(10, "n_msgs_total").iterrows():
    ax.annotate(r["thread_name"], (r["energy"], r["intent"]),
                textcoords="offset points", xytext=(4, 3), fontsize=7, color="#223")
for _, r in df_feats[df_feats["is_group"]==1].iterrows():
    ax.annotate(r["thread_name"], (r["energy"], r["intent"]),
                textcoords="offset points", xytext=(4, 3), fontsize=7, color="#a04040")

ax.set_xlabel("Energy  ←  restrained · · · high-engage  →", fontsize=10)
ax.set_ylabel("Intent  ←  content/task · · · connect/social  →", fontsize=10)
ax.set_xlim(-ax_lim-0.5, ax_lim+0.5)
ax.set_ylim(-ax_lim-0.5, ax_lim+0.5)

legend_patches = [mpatches.Patch(color=MASK_COLORS[i], label=mask_labels[i]) for i in cluster_ids]
legend_patches += [plt.scatter([],[], marker="o", c="#999", s=40, label="DM"),
                   plt.scatter([],[], marker="s", c="#999", s=40, label="Group (red label)")]
ax.legend(handles=legend_patches, loc="upper left", bbox_to_anchor=(1, 1), fontsize=8)
ax.set_title("Social Mask Map\nAxes = lexical proxies  ·  Basins = KDE per cluster  ·  size ~ msg count", fontsize=11)
plt.tight_layout(); plt.show()

## 4. Relationship Drift — Trajectory per Contact

Same quadrant space. Track each DM contact's yearly centroid as an arrow path.
Color = dominant mask in that year.

In [ ]:
DRIFT_CONTACTS = 8
DRIFT_PER_YEAR = 200

_canon_dm = (
    df_feats[df_feats["is_group"] == 0]
    .sort_values("n_msgs_total", ascending=False)
    .head(DRIFT_CONTACTS)["thread_name"].tolist()
)
_canon_thread_ids = {label: meta["thread_ids"] for label, meta in identity_groups.items()}

MASK_TRAJECTORIES = {}
PERSON_COLORS = plt.cm.tab10(np.linspace(0, 0.9, len(_canon_dm)))
fig, ax = plt.subplots(figsize=(11, 10))
ax.axhline(0, color="#eee", lw=0.8); ax.axvline(0, color="#eee", lw=0.8)

all_pts_x, all_pts_y = [], []

for contact, pcolor in zip(_canon_dm, PERSON_COLORS):
    tids = _canon_thread_ids.get(contact, [])
    if not tids:
        continue
    ph = ",".join("?" * len(tids))
    raw_rows = conn.execute(
        f"SELECT timestamp_ms, sender, content, "
        f"STRFTIME('%Y', timestamp_ms/1000,'unixepoch') yr "
        f"FROM messages WHERE thread_id IN ({ph}) AND sender IN ({YOU}) "
        f"AND content_type='text' AND LENGTH(content)>5 ORDER BY timestamp_ms",
        tids
    ).fetchall()

    from itertools import groupby
    rows_by_year = {}
    for r in raw_rows:
        yr = r[3]
        rows_by_year.setdefault(yr, []).append((r[0], r[1], r[2]))

    traj = []
    for yr, yr_rows in sorted(rows_by_year.items()):
        bursted, _ = _fetch_and_burst(tids, sender="you", limit=DRIFT_PER_YEAR)
        # per-year: re-burst just this year's rows directly
        yr_bursted = _burst_aggregate(yr_rows)
        if len(yr_bursted) > DRIFT_PER_YEAR:
            rng = np.random.default_rng(42)
            bucket = len(yr_bursted) / DRIFT_PER_YEAR
            yr_bursted = [yr_bursted[int(rng.integers(int(i*bucket), max(int((i+1)*bucket), int(i*bucket)+1)))]
                          for i in range(DRIFT_PER_YEAR)]
        if len(yr_bursted) < 5:
            continue
        feat_df = pd.DataFrame([_extract_features(yr_bursted)])
        for col in _ALL_FEAT_COLS:
            if col not in feat_df.columns:
                feat_df[col] = 0.0
        e_arr, i_arr = _axis_scores(feat_df, _scaler, _qt_e, _qt_i)
        e_val, i_val = float(e_arr[0]), float(i_arr[0])
        dom = int(_kmeans_2d.predict([[e_val, i_val]])[0])
        traj.append((yr, e_val, i_val, dom))

    if len(traj) < 2:
        continue

    MASK_TRAJECTORIES[contact] = traj
    xs = [t[1] for t in traj]; ys = [t[2] for t in traj]
    all_pts_x.extend(xs); all_pts_y.extend(ys)

    for k in range(len(traj)-1):
        ax.annotate("", xy=(xs[k+1], ys[k+1]), xytext=(xs[k], ys[k]),
                    arrowprops=dict(arrowstyle="->", color=pcolor, lw=1.5))
    for yr, x, y, dom in traj:
        ax.scatter(x, y, color=MASK_COLORS[dom], s=70, zorder=5, edgecolors=pcolor, linewidths=1.8)
        ax.text(x, y, f" {yr}", fontsize=6.5, color=pcolor, va="center")
    ax.scatter(xs[0], ys[0], color=pcolor, s=130, marker="o", zorder=6, label=contact)

person_legend = ax.legend(loc="lower right", fontsize=8, title="Contact")
mask_patches  = [mpatches.Patch(color=MASK_COLORS[i], label=mask_labels[i]) for i in cluster_ids]
ax.add_artist(person_legend)
ax.legend(handles=mask_patches, loc="upper left", bbox_to_anchor=(1, 1), fontsize=8, title="Mask (dot fill)")

if all_pts_x:
    pad = max(max(all_pts_x)-min(all_pts_x), max(all_pts_y)-min(all_pts_y)) * 0.1 + 0.05
    ax.set_xlim(min(all_pts_x)-pad, max(all_pts_x)+pad)
    ax.set_ylim(min(all_pts_y)-pad, max(all_pts_y)+pad)

ax.set_xlabel("Energy  ←  restrained · · · high-engage  →", fontsize=10)
ax.set_ylabel("Intent  ←  content/task · · · connect/social  →", fontsize=10)
ax.set_title("Relationship Drift — Yearly Centroid\nArrow = person  ·  dot fill = dominant mask that year", fontsize=11)
plt.tight_layout(); plt.show()

## 4a. Rolling Lexical/Stylistic Drift

Monthly-bucketed style features (reusing `_extract_features` from Section 3) with a rolling mean and rolling std per contact -- a smoothed trajectory instead of Section 4s single yearly point estimates.

In [ ]:
ROLL_WINDOW = 3  # months
STYLE_DRIFT_FEATS = ["avg_len", "emoji_rate", "func_rate", "sent_strength"]

def _monthly_feature_series(tids, sender="you"):
    ph = ",".join("?" * len(tids))
    sender_clause = f"AND sender IN ({YOU})" if sender == "you" else f"AND sender NOT IN ({YOU})"
    rows = conn.execute(
        f"SELECT timestamp_ms, sender, content, "
        f"STRFTIME('%Y-%m', timestamp_ms/1000,'unixepoch') ym "
        f"FROM messages WHERE thread_id IN ({ph}) {sender_clause} "
        f"AND content_type='text' AND LENGTH(content)>5 ORDER BY timestamp_ms",
        tids
    ).fetchall()
    rows_by_month = {}
    for ts, snd, content, ym in rows:
        rows_by_month.setdefault(ym, []).append((ts, snd, content))
    recs = []
    for ym, mrows in sorted(rows_by_month.items()):
        bursted = _burst_aggregate(mrows)
        if len(bursted) < 5:
            continue
        f = _extract_features(bursted)
        f["month"] = ym
        recs.append(f)
    if not recs:
        return None
    df = pd.DataFrame(recs).set_index("month")
    df.index = pd.to_datetime(df.index)
    return df

STYLE_DRIFT = {}
fig, axes = plt.subplots(len(STYLE_DRIFT_FEATS), 1, figsize=(11, 3*len(STYLE_DRIFT_FEATS)), sharex=True)
for contact, pcolor in zip(_canon_dm, PERSON_COLORS):
    tids = _canon_thread_ids.get(contact, [])
    if not tids:
        continue
    df_m = _monthly_feature_series(tids, sender="you")
    if df_m is None or len(df_m) < ROLL_WINDOW:
        continue
    STYLE_DRIFT[contact] = df_m
    for ax, feat in zip(axes, STYLE_DRIFT_FEATS):
        roll_mean = df_m[feat].rolling(ROLL_WINDOW, min_periods=1).mean()
        roll_std  = df_m[feat].rolling(ROLL_WINDOW, min_periods=1).std().fillna(0)
        ax.plot(df_m.index, roll_mean, color=pcolor, lw=1.5, label=contact)
        ax.fill_between(df_m.index, roll_mean - roll_std, roll_mean + roll_std, color=pcolor, alpha=0.12)

for ax, feat in zip(axes, STYLE_DRIFT_FEATS):
    ax.set_title(f"{feat} -- {ROLL_WINDOW}-month rolling mean +/- std", fontsize=10)
    ax.tick_params(labelsize=8)
axes[0].legend(fontsize=7, ncol=4, loc="upper right")
plt.tight_layout(); plt.show()

## 5. Sentiment: You vs Them (VADER)

In [ ]:
SENT_CONTACTS = 8

sent_contacts = (
    df_feats[df_feats["is_group"] == 0]
    .nlargest(SENT_CONTACTS, "n_msgs_total")["thread_name"].tolist()
)

def _score_burst(tids, is_you):
    texts, _ = _fetch_and_burst(tids, sender="you" if is_you else "them")  # adaptive
    if not texts:
        return {"pos": 0, "neu": 0, "neg": 0, "compound": 0}
    scores = [_sentiment_score(t) for t in texts]
    compound_mean = float(np.mean(scores))
    pos  = float(np.mean([max(s, 0) for s in scores]))
    neg  = float(np.mean([abs(min(s, 0)) for s in scores]))
    return {"pos": pos, "neu": 1 - pos - neg, "neg": neg, "compound": compound_mean}

rows_you  = [_score_burst(_canon_thread_ids.get(c, []), True)  for c in sent_contacts]
rows_them = [_score_burst(_canon_thread_ids.get(c, []), False) for c in sent_contacts]
df_you  = pd.DataFrame(rows_you,  index=sent_contacts)
df_them = pd.DataFrame(rows_them, index=sent_contacts)

fig, ax2 = plt.subplots(figsize=(9, max(4, len(sent_contacts)*0.5)))
y = np.arange(len(sent_contacts))
w = 0.35
ax2.barh(y-w/2, df_you["compound"],  w, label="You",  color="#4a90d9")
ax2.barh(y+w/2, df_them["compound"], w, label="Them", color="#e07b4a")
ax2.set_yticks(y); ax2.set_yticklabels(sent_contacts, fontsize=9)
ax2.axvline(0, color="#ccc", lw=0.8)
ax2.set_xlabel("Compound sentiment  (-1 = negative, +1 = positive)")
ax2.set_title("Overall Tone: You vs Them  (VADER for EN, SnowNLP for CJK)")
ax2.legend(); ax2.invert_yaxis()
plt.tight_layout(); plt.show()

## 5a. Sentiment Trajectory and Changepoint Detection

Monthly rolling sentiment (`_sentiment_score` compound) per contact, with formal changepoint detection (`ruptures` PELT) marking where tone structurally shifted -- instead of eyeballing Section 5s bar chart.

In [ ]:
try:
    import ruptures as rpt
    _RPT_OK = True
except ImportError:
    _RPT_OK = False
    print("ruptures not installed -- changepoint detection will be skipped (pip install ruptures)")

SENT_ROLL_WINDOW = 3  # months
PEN = 1.5  # PELT penalty; higher = fewer detected changepoints

def _monthly_sentiment_series(tids, sender="you"):
    ph = ",".join("?" * len(tids))
    sender_clause = f"AND sender IN ({YOU})" if sender == "you" else f"AND sender NOT IN ({YOU})"
    rows = conn.execute(
        f"SELECT timestamp_ms, sender, content, "
        f"STRFTIME('%Y-%m', timestamp_ms/1000,'unixepoch') ym "
        f"FROM messages WHERE thread_id IN ({ph}) {sender_clause} "
        f"AND content_type='text' AND LENGTH(content)>5 ORDER BY timestamp_ms",
        tids
    ).fetchall()
    rows_by_month = {}
    for ts, snd, content, ym in rows:
        rows_by_month.setdefault(ym, []).append(content)
    recs = []
    for ym, texts in sorted(rows_by_month.items()):
        if len(texts) < 5:
            continue
        scores = [_sentiment_score(t) for t in texts[:200]]
        recs.append({"month": ym, "compound": float(np.mean(scores))})
    if not recs:
        return None
    s = pd.DataFrame(recs).set_index("month")["compound"]
    s.index = pd.to_datetime(s.index)
    return s

SENTIMENT_SERIES = {}
SENTIMENT_CHANGEPOINTS = {}
fig, axes = plt.subplots(len(sent_contacts), 1, figsize=(11, 2.3*len(sent_contacts)), sharex=True)
if len(sent_contacts) == 1:
    axes = [axes]

for ax, contact in zip(axes, sent_contacts):
    tids = _canon_thread_ids.get(contact, [])
    s = _monthly_sentiment_series(tids, sender="you")
    if s is None or len(s) < SENT_ROLL_WINDOW:
        ax.set_visible(False)
        continue
    SENTIMENT_SERIES[contact] = s
    roll = s.rolling(SENT_ROLL_WINDOW, min_periods=1).mean()
    ax.plot(s.index, s.values, color="#ccc", lw=0.8, label="monthly")
    ax.plot(roll.index, roll.values, color="#4a90d9", lw=1.8, label=f"{SENT_ROLL_WINDOW}mo rolling")
    ax.axhline(0, color="#eee", lw=0.6)
    bkps = []
    if _RPT_OK and len(s) >= 6:
        algo = rpt.Pelt(model="rbf").fit(s.values.reshape(-1, 1))
        bkps = algo.predict(pen=PEN)[:-1]  # drop trailing endpoint
        for bp in bkps:
            ax.axvline(s.index[bp], color="#e07b4a", lw=1.2, ls="--")
    SENTIMENT_CHANGEPOINTS[contact] = [str(s.index[bp].date()) for bp in bkps]
    ax.set_ylabel(contact, fontsize=8, rotation=0, ha="right", va="center")
    ax.tick_params(labelsize=7)

axes[0].legend(fontsize=7, loc="upper right")
plt.suptitle("Sentiment Trajectory -- monthly compound, rolling mean, PELT changepoints (dashed)", fontsize=11)
plt.tight_layout(); plt.show()

## 5b. Embedding-Space Drift (BLOC incremental centroid)

Each months sampled messages are embedded with the already-loaded `all-MiniLM-L6-v2` model and folded into a single running centroid via `blocKit.add_to_cluster` -- the same weighted-average incremental-centroid math BLOC uses for belief clusters, reused here with one cluster per month. Cosine distance between consecutive months centroids tracks topical/stylistic drift (rising = drifting, falling = stabilizing).

In [ ]:
from blocKit import add_to_cluster

EMBED_ROLL_CONTACTS = 5
EMBED_PER_MONTH = 40  # cap messages embedded per month per contact (cost control)

def _monthly_centroid_drift(tids, sender="you"):
    ph = ",".join("?" * len(tids))
    sender_clause = f"AND sender IN ({YOU})" if sender == "you" else f"AND sender NOT IN ({YOU})"
    rows = conn.execute(
        f"SELECT timestamp_ms, sender, content, "
        f"STRFTIME('%Y-%m', timestamp_ms/1000,'unixepoch') ym "
        f"FROM messages WHERE thread_id IN ({ph}) {sender_clause} "
        f"AND content_type='text' AND LENGTH(content)>5 ORDER BY timestamp_ms",
        tids
    ).fetchall()
    rows_by_month = {}
    for ts, snd, content, ym in rows:
        rows_by_month.setdefault(ym, []).append(content)

    rng = np.random.default_rng(42)
    months, centroids = [], []
    for ym, texts in sorted(rows_by_month.items()):
        if len(texts) < 5:
            continue
        sample = texts if len(texts) <= EMBED_PER_MONTH else list(rng.choice(texts, EMBED_PER_MONTH, replace=False))
        vecs = _model.encode(sample, normalize_embeddings=True, show_progress_bar=False)
        # BLOC incremental centroid: fold every message of the month into one running cluster.
        # threshold=-1 forces every message to match the (only) existing cluster instead of
        # spawning new ones, so tag_pool[0] ends up as the months weighted-average centroid.
        tag_pool = []
        for v in vecs:
            add_to_cluster(memory={"content": v, "importance": 1.0}, tag_pool=tag_pool, threshold=-1.0)
        months.append(ym)
        centroids.append(tag_pool[0]["centroid"])

    if len(centroids) < 2:
        return None
    dists = [np.nan] + [float(1 - np.dot(centroids[i - 1], centroids[i]))
                         for i in range(1, len(centroids))]
    return pd.DataFrame({"month": pd.to_datetime(months), "cos_dist_from_prev": dists}).set_index("month")

EMBED_DRIFT = {}
fig, ax = plt.subplots(figsize=(11, 5))
for contact, pcolor in zip(_canon_dm[:EMBED_ROLL_CONTACTS], PERSON_COLORS):
    tids = _canon_thread_ids.get(contact, [])
    if not tids:
        continue
    df_d = _monthly_centroid_drift(tids, sender="you")
    if df_d is None:
        continue
    EMBED_DRIFT[contact] = df_d
    ax.plot(df_d.index, df_d["cos_dist_from_prev"], marker="o", ms=3, lw=1.3, color=pcolor, label=contact)

ax.set_ylabel("cosine distance: centroid(T) vs centroid(T-1)")
ax.set_title("Embedding-Space Drift -- monthly message centroid (all-MiniLM-L6-v2, BLOC incremental centroid)", fontsize=11)
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 6. Linguistic Mirroring

For each top DM contact, compute the same style features for **your** messages and **their** messages separately, then measure how aligned they are.

- **Mirror score** (0–1): inverse of mean normalized delta across features — higher = you mirror them more
- **Lexical entrainment**: % of your vocabulary that overlaps with theirs (vs. your global baseline)
- **Feature deltas**: which specific axis you converge/diverge on most

In [ ]:
MIRROR_CONTACTS = 15

_MIRROR_FEATS = ["avg_len", "emoji_rate", "question_rate", "excl_rate", "caps_ratio", "func_rate", "cjk_ratio"]

def _style_vec(texts):
    if not texts:
        return None
    n  = len(texts)
    wc = [len(t.split()) for t in texts]
    total_chars = sum(len(t) for t in texts)
    return {
        "avg_len":       float(np.mean(wc)),
        "emoji_rate":    sum(len(_EMOJI_RE.findall(t)) for t in texts) / n,
        "question_rate": sum(1 for t in texts if "?" in t or "？" in t) / n,
        "excl_rate":     sum(t.count("!") for t in texts) / n,
        "caps_ratio":    sum(sum(c.isupper() for c in t) / max(len(t), 1) for t in texts) / n,
        "func_rate":     sum(len(_FUNC_RE.findall(t)) for t in texts) / max(sum(wc), 1),
        "cjk_ratio":     sum(len(_CJK_RE.findall(t)) for t in texts) / max(total_chars, 1),
    }

# Global your-vocabulary baseline — fixed large sample across all threads
_baseline_texts, _ = _fetch_and_burst(
    [tid for meta in identity_groups.values() for tid in meta["thread_ids"]],
    sender="you", limit=5000
)
_all_your_words = set(w for t in _baseline_texts for w in t.lower().split())

top_dms = df_feats[df_feats["is_group"]==0].nlargest(MIRROR_CONTACTS, "n_msgs_total")["thread_name"].tolist()

mirror_rows = []
for contact in top_dms:
    tids = _canon_thread_ids.get(contact, [])
    if not tids:
        continue

    you_texts,  _ = _fetch_and_burst(tids, sender="you")   # adaptive per contact
    them_texts, _ = _fetch_and_burst(tids, sender="them")  # adaptive per contact

    if len(you_texts) < 10 or len(them_texts) < 10:
        continue

    fy = _style_vec(you_texts)
    ft = _style_vec(them_texts)

    deltas = {f: abs(fy[f] - ft[f]) / max(abs(fy[f]), abs(ft[f]), 1e-4) for f in _MIRROR_FEATS}
    mirror_score = float(1 - np.mean(list(deltas.values())))

    your_words  = set(w for t in you_texts  for w in t.lower().split())
    their_words = set(w for t in them_texts for w in t.lower().split())
    entrainment_raw      = len(your_words & their_words) / max(len(your_words), 1)
    entrainment_baseline = len(_all_your_words & their_words) / max(len(_all_your_words), 1)
    entrainment_lift     = entrainment_raw - entrainment_baseline

    row = {"contact": contact, "mirror_score": mirror_score, "entrainment_lift": entrainment_lift}
    for f in _MIRROR_FEATS:
        row[f"you_{f}"]   = fy[f]
        row[f"them_{f}"]  = ft[f]
        row[f"delta_{f}"] = deltas[f]
    mirror_rows.append(row)

df_mirror = pd.DataFrame(mirror_rows).sort_values("mirror_score", ascending=False).reset_index(drop=True)

# ── Plot 1: Mirror score + entrainment lift ───────────────────────────────────
fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(14, max(4, len(df_mirror)*0.45)))
contacts_ord = df_mirror["contact"].tolist()
y = np.arange(len(contacts_ord))

ax_a.barh(y, df_mirror["mirror_score"], color="#5b8dd9", alpha=0.85)
ax_a.set_yticks(y); ax_a.set_yticklabels(contacts_ord, fontsize=8)
ax_a.set_xlabel("Mirror score  (0 = no match, 1 = perfect match)")
ax_a.set_title("Linguistic Mirroring Score")
ax_a.axvline(df_mirror["mirror_score"].mean(), color="#e07b4a", lw=1.2, ls="--", label="mean")
ax_a.legend(fontsize=8); ax_a.invert_yaxis()

colors_lift = ["#74c476" if v >= 0 else "#e07b4a" for v in df_mirror["entrainment_lift"]]
ax_b.barh(y, df_mirror["entrainment_lift"], color=colors_lift, alpha=0.85)
ax_b.set_yticks(y); ax_b.set_yticklabels(contacts_ord, fontsize=8)
ax_b.axvline(0, color="#ccc", lw=0.8)
ax_b.set_xlabel("Lexical entrainment lift  (above your global baseline)")
ax_b.set_title("Vocabulary Adoption (beyond baseline)")
ax_b.invert_yaxis()

plt.suptitle("Linguistic Mirroring — top DM contacts", fontsize=12)
plt.tight_layout(); plt.show()

# ── Plot 2: Feature delta heatmap ─────────────────────────────────────────────
delta_cols = [f"delta_{f}" for f in _MIRROR_FEATS]
delta_mat  = df_mirror.set_index("contact")[delta_cols].values

fig, ax_h = plt.subplots(figsize=(len(_MIRROR_FEATS)*1.2 + 1, max(4, len(df_mirror)*0.4)))
im = ax_h.imshow(delta_mat, aspect="auto", cmap="YlOrRd", vmin=0, vmax=1)
ax_h.set_xticks(range(len(_MIRROR_FEATS)))
ax_h.set_xticklabels([f.replace("_rate","").replace("_ratio","") for f in _MIRROR_FEATS], fontsize=9)
ax_h.set_yticks(range(len(contacts_ord)))
ax_h.set_yticklabels(contacts_ord, fontsize=8)
plt.colorbar(im, ax=ax_h, label="Normalized delta  (0=identical, 1=very different)")
ax_h.set_title("Which features you diverge from most — per contact\n(dark = you talk very differently to them on this axis)", fontsize=10)
plt.tight_layout(); plt.show()

## 7. Relationship Dynamics

Structural features derived from timestamps and sender patterns — independent of what you write, capturing **how** the relationship operates.

- **Initiation rate**: share of conversation sessions you start (>0.5 = you pursue more)
- **Response advantage**: `log(their latency / your latency)` — positive = you reply faster = more engaged
- **Session frequency**: conversations per month over the active period
- **Recency trend**: message volume last 3 months vs 3–6 months ago (+ = growing, − = fading)
- **Call ratio**: calls per session (higher = more intimate modality)
- **Late-night ratio**: share of messages sent 11pm–4am (intimacy / availability signal)

**Main scatter** — quadrants read as:
- Top-right: you initiate + you reply faster → **fully pursuing**
- Top-left: they initiate + you reply faster → **receptive / they come to you**
- Bottom-left: they initiate + you reply slower → **detached, they chase**
- Bottom-right: you initiate + you reply slower → **reaching out but less engaged**

In [ ]:
import datetime

RELDY_CONTACTS = 15
SESSION_GAP    = 3 * 3600   # seconds — gap that marks a new conversation session
MAX_REPLY_GAP  = 24 * 3600  # ignore reply latencies > 24 h (stale / forgotten)

top_reldyn = (
    df_feats[df_feats["is_group"] == 0]
    .nlargest(RELDY_CONTACTS, "n_msgs_total")["thread_name"].tolist()
)

# Use real current time so recency window is always relative to today
_NOW_TS = datetime.datetime.now().timestamp()

reldyn_rows = []
for contact in top_reldyn:
    tids = _canon_thread_ids.get(contact, [])
    if not tids:
        continue
    ph = ",".join("?" * len(tids))
    rows = conn.execute(
        f"SELECT timestamp_ms, sender, content_type, call_duration "
        f"FROM messages WHERE thread_id IN ({ph}) "
        f"AND content_type IN ('text','call') ORDER BY timestamp_ms",
        tids,
    ).fetchall()
    if len(rows) < 20:
        continue

    msgs = [(r[0] / 1000, r[1] in YOUR_NAMES, r[2], r[3]) for r in rows]
    text_msgs = [(ts, iy) for ts, iy, ct, _ in msgs if ct == "text"]

    # ── Initiation rate ───────────────────────────────────────────────────────
    sessions_you = sessions_them = 0
    prev_ts = None
    for ts, is_you in text_msgs:
        if prev_ts is None or (ts - prev_ts) > SESSION_GAP:
            if is_you:
                sessions_you += 1
            else:
                sessions_them += 1
        prev_ts = ts
    total_sessions = sessions_you + sessions_them
    initiation_rate = sessions_you / max(total_sessions, 1)

    # ── Response latency ──────────────────────────────────────────────────────
    your_latencies, their_latencies = [], []
    for i in range(len(text_msgs) - 1):
        ts0, iy0 = text_msgs[i]
        ts1, iy1 = text_msgs[i + 1]
        gap = ts1 - ts0
        if gap > MAX_REPLY_GAP or gap < 1:
            continue
        if iy0 and not iy1:
            their_latencies.append(gap)
        elif not iy0 and iy1:
            your_latencies.append(gap)

    your_med  = float(np.median(your_latencies))  if your_latencies  else np.nan
    their_med = float(np.median(their_latencies)) if their_latencies else np.nan
    resp_adv  = float(np.log(their_med / your_med)) if (your_med > 0 and their_med > 0) else 0.0

    # ── Session frequency (sessions / month over active window) ───────────────
    first_ts, last_ts = msgs[0][0], msgs[-1][0]
    active_months = max((last_ts - first_ts) / (30 * 86400), 1)
    session_freq  = total_sessions / active_months

    # ── Recency trend: last 3 mo vs 3–6 mo ago, relative to TODAY ────────────
    three_mo = _NOW_TS - 90  * 86400
    six_mo   = _NOW_TS - 180 * 86400
    cnt_0_3  = sum(1 for ts, _, ct, _ in msgs if ts >= three_mo and ct == "text")
    cnt_3_6  = sum(1 for ts, _, ct, _ in msgs if six_mo <= ts < three_mo and ct == "text")
    # Bounded to [-1, +1] — avoids explosion when one window has near-zero messages
    recency_trend = (cnt_0_3 - cnt_3_6) / max(cnt_0_3 + cnt_3_6, 1)

    # ── Call ratio ────────────────────────────────────────────────────────────
    n_calls    = sum(1 for _, _, ct, _ in msgs if ct == "call")
    call_ratio = n_calls / max(total_sessions, 1)

    # ── Late-night ratio (23:00–04:00 local) ─────────────────────────────────
    late = sum(
        1 for ts, _, ct, _ in msgs
        if ct == "text" and (datetime.datetime.fromtimestamp(ts).hour >= 23
                              or datetime.datetime.fromtimestamp(ts).hour < 4)
    )
    late_night_ratio = late / max(len(text_msgs), 1)

    reldyn_rows.append({
        "contact":           contact,
        "initiation_rate":   initiation_rate,
        "resp_adv":          resp_adv,
        "your_latency_min":  your_med  / 60 if not np.isnan(your_med)  else np.nan,
        "their_latency_min": their_med / 60 if not np.isnan(their_med) else np.nan,
        "session_freq":      session_freq,
        "recency_trend":     recency_trend,
        "call_ratio":        call_ratio,
        "late_night_ratio":  late_night_ratio,
        "total_sessions":    total_sessions,
    })

df_reldyn = pd.DataFrame(reldyn_rows)

print("── Relationship Dynamics ─────────────────────────────────────────────────")
display_cols = ["contact","initiation_rate","resp_adv","your_latency_min",
                "their_latency_min","session_freq","recency_trend","call_ratio","late_night_ratio"]
print(df_reldyn[display_cols].to_string(index=False, float_format=lambda x: f"{x:.2f}"))

# ── Plot 1: Initiation vs Response Advantage ──────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 9))
ax.axhline(0,   color="#ddd", lw=0.8)
ax.axvline(0.5, color="#ddd", lw=0.8)

resp_max = df_reldyn["resp_adv"].abs().max() or 1
for sx, sy, lbl in [(1,1,"you initiate\nyou reply faster"), (-1,1,"they initiate\nyou reply faster"),
                     (1,-1,"you initiate\nthey reply faster"), (-1,-1,"they initiate\nthey reply faster")]:
    ax.text(0.5 + sx*0.23, sy*0.82*resp_max,
            lbl, ha="center", fontsize=7.5, color="#bbb", style="italic", va="center")

sz   = np.clip(df_reldyn["session_freq"] * 18, 40, 500)
cval = df_reldyn["recency_trend"].clip(-1, 1)
sc   = ax.scatter(
    df_reldyn["initiation_rate"], df_reldyn["resp_adv"],
    s=sz, c=cval, cmap=plt.cm.RdYlGn, vmin=-1, vmax=1,
    edgecolors="white", linewidths=0.8, zorder=5, alpha=0.9,
)
for _, r in df_reldyn.iterrows():
    ax.annotate(r["contact"], (r["initiation_rate"], r["resp_adv"]),
                textcoords="offset points", xytext=(5, 3), fontsize=7.5, color="#223")

plt.colorbar(sc, ax=ax, label="Recency trend  (green = growing, red = fading)")
ax.set_xlabel("Initiation rate  ←  they always start · · · you always start  →", fontsize=10)
ax.set_ylabel("Response advantage  log(their latency / your latency)\n←  they reply faster · · · you reply faster  →", fontsize=10)
ax.set_title("Relationship Dynamics — Initiation × Engagement\nDot size = session frequency  ·  color = recency trend", fontsize=11)
plt.tight_layout(); plt.show()

# ── Plot 2: Feature profile bars ──────────────────────────────────────────────
_RD_FEATS  = ["initiation_rate","resp_adv","session_freq","recency_trend","call_ratio","late_night_ratio"]
_RD_LABELS = {
    "initiation_rate":  "You initiate (rate)",
    "resp_adv":         "Response advantage (log)",
    "session_freq":     "Session freq / month",
    "recency_trend":    "Recency trend  [-1, +1]",
    "call_ratio":       "Call ratio",
    "late_night_ratio": "Late-night ratio",
}

df_rd_sorted = df_reldyn.sort_values("initiation_rate", ascending=False).reset_index(drop=True)
contacts_ord = df_rd_sorted["contact"].tolist()
y = np.arange(len(contacts_ord))

fig, axes = plt.subplots(1, len(_RD_FEATS), figsize=(16, max(4, len(df_rd_sorted)*0.38)))
for ax_f, feat in zip(axes, _RD_FEATS):
    vals   = df_rd_sorted[feat].fillna(0).values
    colors = ["#74c476" if v >= 0 else "#e07b4a"
              if feat in ("recency_trend","resp_adv") else "#5b8dd9"
              for v in vals]
    ax_f.barh(y, vals, color=colors, alpha=0.85)
    ax_f.set_yticks(y)
    ax_f.set_yticklabels(contacts_ord if ax_f is axes[0] else [], fontsize=8)
    ax_f.axvline(0, color="#ccc", lw=0.7)
    ax_f.set_title(_RD_LABELS[feat], fontsize=7.5)
    ax_f.tick_params(axis="both", labelsize=7)
    ax_f.invert_yaxis()

plt.suptitle("Relationship Dynamics — per contact profile", fontsize=11)
plt.tight_layout(); plt.show()

# ── Latency table ─────────────────────────────────────────────────────────────
print("\n── Response latency summary (median minutes) ─────────────────────────────")
for _, r in df_reldyn.sort_values("resp_adv", ascending=False).iterrows():
    you_str  = f"{r['your_latency_min']:.1f} min"  if not np.isnan(r['your_latency_min'])  else "n/a"
    them_str = f"{r['their_latency_min']:.1f} min" if not np.isnan(r['their_latency_min']) else "n/a"
    adv = "you faster" if r["resp_adv"] > 0.1 else ("equal" if abs(r["resp_adv"]) <= 0.1 else "they faster")
    print(f"  {r['contact']:20s}  you={you_str:10s}  them={them_str:10s}  → {adv}")

## 8. LLM-Narrated Personality & Style Evolution

Sections 4/4a/5a/5b establish *that* and *how much* your communication style, sentiment, and topical embedding drifted per contact -- a rolling mean, a PELT changepoint, a cosine distance are inherently quantitative and can't say *why* something changed or characterize it in human terms ("became more formal", "became more withdrawn"). This section hands the already-computed quantitative summaries (never raw messages) to an LLM via `litellm` + `instructor`, asking it to narrate the shifts in plain language.

**Tradeoff, stated plainly**: everything above this section is deterministic and reproducible -- rerun it and you get the same numbers. The LLM narration below is qualitative and unverifiable, and can produce a plausible-sounding story that doesn't actually match the underlying numbers. Treat it as a hypothesis-generation aid to read alongside the charts above, not as ground truth. It's deliberately fed *summary statistics*, not raw content, so it can't invent specific events or topics -- but it can still misjudge magnitude or overstate confidence.

In [ ]:
import json
import litellm
import instructor
from pydantic import BaseModel, Field
from typing import List

LLM_MODEL = "gpt-4.1"


class ContactShift(BaseModel):
    contact: str
    narrative: str = Field(..., description="1-3 sentence plain-language description of how this person's communication with you evolved, grounded only in the provided numbers")
    confidence: float = Field(..., ge=0, le=1, description="Model's own confidence that the narrative is well-supported by the numeric evidence given -- not a measure of ground truth")


class EvolutionReport(BaseModel):
    overall_summary: str = Field(..., description="2-4 sentence summary across all contacts")
    per_contact: List[ContactShift]
    caveats: str = Field(..., description="What this narration can't verify or might get wrong, given only summarized stats as input")


def _series_trend(s):
    if s is None or len(s) < 2:
        return None
    return {
        "start": round(float(s.iloc[0]), 4),
        "end": round(float(s.iloc[-1]), 4),
        "n_points": len(s),
        "first_month": str(s.index[0].date()),
        "last_month": str(s.index[-1].date()),
    }


def _build_contact_summary(contact):
    summary = {"contact": contact}

    style_df = STYLE_DRIFT.get(contact)
    if style_df is not None and len(style_df):
        summary["style_features"] = {
            feat: _series_trend(style_df[feat]) for feat in STYLE_DRIFT_FEATS if feat in style_df
        }

    sent_s = SENTIMENT_SERIES.get(contact)
    if sent_s is not None:
        summary["sentiment"] = _series_trend(sent_s)
        summary["sentiment_changepoints"] = SENTIMENT_CHANGEPOINTS.get(contact, [])

    embed_df = EMBED_DRIFT.get(contact)
    if embed_df is not None and len(embed_df):
        d = embed_df["cos_dist_from_prev"].dropna()
        if len(d):
            summary["embedding_drift"] = {
                "mean_monthly_cos_dist": round(float(d.mean()), 4),
                "max_cos_dist": round(float(d.max()), 4),
                "max_cos_dist_month": str(d.idxmax().date()),
            }

    traj = MASK_TRAJECTORIES.get(contact)
    if traj:
        summary["mask_trajectory"] = [{"year": yr, "dominant_mask": mask_labels[dom]} for yr, _, _, dom in traj]

    return summary


_llm_contacts = [c for c in _canon_dm if c in STYLE_DRIFT or c in SENTIMENT_SERIES or c in EMBED_DRIFT]
contact_summaries = [_build_contact_summary(c) for c in _llm_contacts]

client = instructor.from_litellm(litellm.completion)

report = client.chat.completions.create(
    model=LLM_MODEL,
    response_model=EvolutionReport,
    messages=[
        {"role": "system", "content": (
            "You are analyzing pre-computed statistical summaries of one person's communication style, "
            "sentiment, and topic-embedding drift with several contacts over multiple years. "
            "You are given ONLY summary statistics, never raw messages. "
            "Ground every claim strictly in the numbers provided -- do not invent details, "
            "specific events, or topics that aren't implied by the stats. "
            "If the numbers are ambiguous or too sparse to support a claim, say so instead of guessing."
        )},
        {"role": "user", "content": (
            "Here are per-contact statistical summaries (rolling style features, sentiment trend and "
            "detected changepoints, embedding-space drift, and yearly dominant social-mask trajectory). "
            "Narrate what changed for each contact and overall, in plain language:\n\n"
            + json.dumps(contact_summaries, indent=2, default=str)
        )},
    ],
)

print("=" * 80)
print("OVERALL SUMMARY")
print("=" * 80)
print(report.overall_summary)
print()
for c in report.per_contact:
    print(f"[{c.contact}]  (confidence: {c.confidence:.2f})")
    print(f"  {c.narrative}\n")
print("=" * 80)
print("CAVEATS")
print("=" * 80)
print(report.caveats)

## 9. Anchored Close-Reading (opt-in, sends raw message samples)

Section 8 narrates from summary statistics only. This section goes one step further for windows the quantitative signals already flagged as anomalous (a PELT sentiment changepoint, or the single largest month-over-month embedding-centroid jump per contact) -- it pulls the small raw message sample from *just that window* and asks the LLM to describe concretely what's different about it.

**This is qualitatively different from Section 8 and from the rejected "freestyle over all raw samples" idea:**
- It only runs on windows a deterministic signal already flagged -- it is not discovering drift from scratch, it is close-reading a statistically-justified slice.
- It still sends real message content (yours) to an external API. That is a materially different privacy posture than Section 8's stats-only payload -- **gated behind `RUN_RAW_SAMPLE_NARRATION`, off by default.**
- Treat the output as an anecdotal hypothesis about one narrow window, not a verified account. Small samples, LLM narration, no ground truth to check against.

In [ ]:
RUN_RAW_SAMPLE_NARRATION = False  # sends raw message content to the LLM API -- opt in, off by default

if not RUN_RAW_SAMPLE_NARRATION:
    print("RUN_RAW_SAMPLE_NARRATION is False -- skipping. Set to True to run this section.")
else:
    import json
    import litellm
    import instructor
    from pydantic import BaseModel, Field

    WINDOW_SAMPLE_SIZE = 25       # max raw messages pulled per flagged window
    EMBED_SPIKE_MIN = 0.15        # ignore embedding spikes below this cosine distance (noise floor)

    class WindowReading(BaseModel):
        contact: str
        month: str
        signal: str = Field(..., description="the quantitative signal that flagged this window")
        observation: str = Field(..., description="concrete description of what is different in this window, grounded only in the quoted messages")
        caveat: str = Field(..., description="why this specific observation might not generalize or might be an artifact of a small sample")

    def _raw_window_sample(tids, month, limit=WINDOW_SAMPLE_SIZE):
        ph = ",".join("?" * len(tids))
        rows = conn.execute(
            f"SELECT timestamp_ms, content, "
            f"STRFTIME('%Y-%m', timestamp_ms/1000,'unixepoch') ym "
            f"FROM messages WHERE thread_id IN ({ph}) AND sender IN ({YOU}) "
            f"AND content_type='text' AND LENGTH(content)>5 "
            f"AND STRFTIME('%Y-%m', timestamp_ms/1000,'unixepoch')=? "
            f"ORDER BY timestamp_ms",
            tids + [month]
        ).fetchall()
        bursted = _burst_aggregate([(r[0], "you", r[1]) for r in rows])
        if len(bursted) > limit:
            rng = np.random.default_rng(42)
            idxs = sorted(rng.choice(len(bursted), limit, replace=False))
            bursted = [bursted[i] for i in idxs]
        return bursted

    def _flagged_windows(contact):
        windows = []
        for month in SENTIMENT_CHANGEPOINTS.get(contact, [])[:1]:
            ym = month[:7]
            windows.append((ym, "PELT sentiment changepoint detected at this month"))
        embed_df = EMBED_DRIFT.get(contact)
        if embed_df is not None and len(embed_df):
            d = embed_df["cos_dist_from_prev"].dropna()
            if len(d) and d.max() >= EMBED_SPIKE_MIN:
                top_month = str(d.idxmax().date())[:7]
                windows.append((top_month, f"largest month-over-month embedding-centroid jump in series (cos_dist={d.max():.3f})"))
        return windows

    client = instructor.from_litellm(litellm.completion)
    readings = []

    for contact in _llm_contacts:
        tids = _canon_thread_ids.get(contact, [])
        if not tids:
            continue
        for month, signal in _flagged_windows(contact):
            texts = _raw_window_sample(tids, month)
            if len(texts) < 5:
                continue
            reading = client.chat.completions.create(
                model=LLM_MODEL,
                response_model=WindowReading,
                messages=[
                    {"role": "system", "content": (
                        "You are close-reading a small sample of one person's own chat messages from a single "
                        "flagged month. Describe concretely what stands out in THIS sample -- tone, topic, "
                        "phrasing, anything notable. Do not generalize beyond this sample or speculate about "
                        "causes you cannot see in the text. Be specific and quote or paraphrase briefly where useful."
                    )},
                    {"role": "user", "content": (
                        f"Contact: {contact}\nMonth: {month}\n"
                        f"Why this window was flagged: {signal}\n\n"
                        "Sampled messages (chronological):\n" + "\n---\n".join(texts)
                    )},
                ],
            )
            readings.append(reading)

    for r in readings:
        print(f"[{r.contact} / {r.month}]  signal: {r.signal}")
        print(f"  observation: {r.observation}")
        print(f"  caveat: {r.caveat}\n")

    if not readings:
        print("No windows met the flagging thresholds -- nothing to narrate.")